# Kabe's Maybes — Project History & Architecture

A full account of how this project was built, every major decision made, and how all the pieces connect.

## What This Is

**Kabe's Maybes** (`kabes-maybes.vercel.app`) is a data journalism–style UFC analytics platform. It scrapes every UFC event from UFCStats.com, trains machine learning models on historical fight outcomes, and publishes weekly predictions for upcoming cards — complete with a transparency scorecard showing past accuracy.

The aesthetic is deliberately "analyst who loves MMA" rather than commercial product: numbers-forward, no marketing copy, dark mode by default.

## How It Started (Commits: `d1e4903` → `8ceb197`)

The project began as a pure data scraping exercise. The earliest commits show an evolving attempt to get UFC data from UFCStats.com — a site with no official API. The first approach used a third-party scraper (`scrape_ufc_stats` added as a git submodule, then immediately removed when it proved broken).

**Key early decisions:**

- **PostgreSQL on Supabase** — chosen for its free tier, built-in SQL editor, and direct `psycopg2` connection. No ORM at this stage — raw SQL inserts.
- **6-table schema** — `event_details`, `fighter_details`, `fight_details`, `fight_results`, `fighter_tott` (Tale of the Tape), `fight_stats` — mirroring exactly what UFCStats exposes.
- **GitHub Actions** for automation from day one — weekly scrape runs on Sunday 18:00 UTC, keepalive pings to prevent Supabase from sleeping.
- **VARCHARs everywhere for IDs** — UFCStats uses hex-style IDs in URLs (e.g. `/fighter-details/3b27d3e9`). These were preserved as-is rather than converting to integers, because they're the natural join keys between the scraper output and the database.

The scraper went through many iterations (`full_historical_scraper`, then `live_scraper.py`) before settling. The final `live_scraper.py` handles all 6 tables in one run.

## Data & ETL Pipeline (Task 3 — `2b689af` → `5e1144b`)

After the initial data load (756 events, 4,449 fighters, 8,482 fights, 39,912 fight stats), the raw data was messy:

| Problem | Example |
|---------|--------|
| Stats as strings | `"17 of 37"` (sig strikes landed / attempted) |
| Time as clock string | `"3:42"` (not seconds) |
| Physical attrs as strings | `"6' 1\""`, `"185 lbs."` |
| Missing data placeholder | `"--"` everywhere |
| No FK relationships | `fight_stats` had no `fighter_id` |
| No derived columns | no weight class, no title fight flags |

**ETL pipeline** (`post_scrape_clean.py`) runs 4 phases in sequence:

1. **FK Resolution** — populate `fighter_a_id`, `fighter_b_id` on fight_details; `fighter_id`/`opponent_id` on fight_results; `fighter_id` on fight_stats (via fuzzy name matching with rapidfuzz, WRatio score ≥88)
2. **Quality Cleanup** — NULL out `"--"` placeholders, strip trailing spaces from METHOD
3. **Type Parsing** — parse `"X of Y"` strings into `sig_str_landed`/`sig_str_attempted` integers; `CTRL` → `ctrl_seconds`; height/weight/reach/dob into typed numeric columns
4. **Derived Columns** — `weight_class` (inferred from WEIGHTCLASS text), `is_title_fight`, `is_interim_title`, `is_championship_rounds`

This runs automatically after every weekly scrape via GitHub Actions chaining:
```
scraper → ETL → feature engineering → model retrain
```

> **Critical query pattern discovered during ETL**: `fight_results` has one row per fight, where `fighter_id` = winner and `opponent_id` = loser. A naive `WHERE fighter_id = :id` returns wins only — losses are invisible. All fighter history queries use an OR pattern (`WHERE fighter_id = :id OR opponent_id = :id`).

## ML Models (Task 6 — `aee098c`)

**Feature engineering**: 30+ features built from fighter differentials:

| Category | Features |
|----------|----------|
| Physical | height diff, reach diff, weight diff, age diff |
| Experience | UFC fights diff, win streak diff, finish rate diff |
| Striking (rolling 5-fight) | sig strike accuracy diff, head/body/leg distribution |
| Grappling | takedown % diff, submission attempt rate diff, control time diff |
| Derived | days since last fight, career KO% diff, career sub% diff |

**Two-model ensemble:**
- **Random Forest** (primary) — `sklearn.ensemble.RandomForestClassifier`, trained on fights from Jan 2017 onward (earlier fights lack reliable stat coverage)
- **XGBoost** — used as a cross-check; RF ensemble is the production predictor

**Method prediction**: separate multi-class model predicting KO/TKO vs Submission vs Decision probability

**Training data**: built into `training_data.parquet` by `feature_engineering.py`, auto-rebuilt by GitHub Actions after each ETL run, then `retrain.yml` retrains and commits updated model artifacts

**Validation results:**

| Metric | Value |
|--------|-------|
| Overall accuracy (1716 test fights, Jan 2022–present) | **62.4%** |
| High-confidence accuracy (≥65% win probability) | **84.2%** |
| Total fights evaluated | 1,716 |
| Correct predictions | 1,071 |

## Backend — FastAPI (Task 4 → Tasks 11–19)

**Tech choice**: FastAPI was chosen over Flask for automatic OpenAPI docs (invaluable for frontend development) and Pydantic v2 validation.

**Structure:**
```
backend/
├── api/
│   ├── main.py              # App, CORS, middleware, exception handlers
│   ├── dependencies.py      # get_db() session dependency
│   └── v1/endpoints/        # One file per feature area
│       ├── fighters.py
│       ├── fights.py
│       ├── events.py
│       ├── predictions.py   # POST /predictions/fight-outcome
│       ├── upcoming.py      # Upcoming events/fights/predictions
│       ├── past_predictions.py
│       └── analytics.py
├── schemas/                 # Pydantic v2 response models
├── features/                # Feature extraction for ML inference
│   └── extractors.py        # get_fights_long_df(), build_feature_vector()
└── scraper/                 # All data pipeline scripts
```

**Middleware**: `RequestIDMiddleware` (adds `X-Request-ID` header), `TimingMiddleware` (logs request duration)

**Key design decisions:**
- All raw SQL via SQLAlchemy `text()` — no ORM models. The schema is legacy/append-only; defining ORM models would add maintenance burden with no benefit.
- Pydantic schemas defined separately from DB access — clean separation between "what the DB returns" and "what the API exposes"
- `get_db()` yields a session per request, closes on completion

> **Important FastAPI routing lesson**: `GET /fights` must be registered *before* `GET /fights/{fight_id}` in the router. FastAPI matches routes in registration order — if `/{fight_id}` is first, the literal path `/fights` gets treated as a fight ID.

## Hosting Odyssey

The backend went through three hosting platforms before settling:

| Stage | Platform | Reason for Change |
|-------|----------|-------------------|
| Initial | Local only | Development |
| v1 | **Fly.io** | First deployment attempt; Docker-based, generous free tier |
| v2 | **Render** | Fly.io free tier was eliminated; Render offers free Web Services |
| v3 ✅ | **Google Cloud Run** | Render free tier spins down after 15 min inactivity (~30s cold start) |

**Frontend**: always Vercel. Connected directly to the GitHub repo — every push to `main` triggers a deploy. Set `Root Directory = frontend` in Vercel settings. One env var: `VITE_API_BASE_URL`.

**Database**: always Supabase (PostgreSQL). Free tier, direct psycopg2 connection string, no changes needed as backend hosting changed.

**Current production stack:**
```
Browser → Vercel (React SPA, CDN) → Google Cloud Run (FastAPI, Docker) → Supabase (PostgreSQL)
```

**Cloud Run setup details:**
- Project: `kabes-maybes` (Google Cloud, personal Gmail account)
- Service: `kabes-maybes-api`, region `us-central1`
- Image: `us-central1-docker.pkg.dev/kabes-maybes/kabes-maybes/api:latest` (Artifact Registry)
- `min-instances=0` (scales to zero, ~1–2s cold start vs Render's ~30s)
- PORT env var passed at runtime — `gunicorn.conf.py` binds to `$PORT` (defaults to 8000)
- Secrets managed via Google Cloud Secret Manager: `DATABASE_URL`, `ALLOWED_ORIGINS`
- Auto-deploy on every push to `main` via GitHub Actions (`deploy-backend.yml`)

**Why not Firebase?** Firebase is a higher-level app-dev platform (Firestore, Auth, Hosting) built on top of GCP. Cloud Run is the right choice here — it runs any Docker container, which is exactly what a FastAPI + Gunicorn app needs. Firebase Hosting is only for static sites.

**Rollback to Render**: one env var change in Vercel (`VITE_API_BASE_URL` → Render URL). No code changes needed.

## Frontend — React (Task 7 → Tasks 21–23)

**Tech choices:**

| Layer | Choice | Why |
|-------|--------|-----|
| Framework | React 19 + TypeScript 5.9 (Vite) | Fast dev server, first-class TS support |
| Styling | Tailwind CSS v4 (CSS-first) | No config file; `@theme` block in `index.css` is the single source of truth |
| Routing | React Router v6 (all lazy-loaded) | Code splitting per route, no upfront bundle cost |
| Charts | Recharts | Simple API, good defaults |
| HTTP | Axios | Interceptors for error normalisation |
| State | Context API + useReducer | No external store needed for this scale |

**Design system**: All colors, fonts, and shadows as CSS custom properties in `frontend/src/index.css` `@theme` block. Components never hardcode values — they reference `var(--color-accent)` etc.

**Dark / light mode**: `ThemeProvider` reads `localStorage` on mount, falls back to `prefers-color-scheme`. Toggle in header. Both modes equally polished.

## Page-by-Page History

### Home (`/`)
Started as a placeholder. Grew to host the **ModelScorecard** — the most complex component on the site. Scorecard shows:
- Compact accuracy stats: `62.4% accurate · 1071/1716 fights (84.2% when ≥65% confident)`
- One-sentence model description
- **Events tab**: browse all past events the model was evaluated on, with per-event accuracy, search + year filter + pagination
- **Fight Search tab**: search any fighter name to see all predictions involving them; defaults to 10 most recent fights

### Upcoming (`/upcoming`)
The centrepiece. Accordion-style: event rows expand to show the full fight card. Per-fight row shows fighter names, win probability bar, method chip. Design iterations visible in git history — went through many layouts (centered name, arrows, badges) before landing on the current clean version where:
- Badges (NEXT / numbered event) are always on their own line above event name
- Winner is indicated by full-contrast name vs muted loser — no arrow needed
- Date + location on line 2, bout count on line 3 — consistent wrapping on all screens

Fight rows link to the matchup page.

### Fight Matchup (`/upcoming/fights/:id` and `/past-predictions/fights/:id`)
Dedicated page per fight. Same layout for both upcoming and past predictions:

```
[Fighter A]                    [Fighter B]
       Win probability bar (dominant visual)
       Method: KO/TKO 32% | Sub 18% | Dec 50%

       Tale of the Tape (height / weight / reach / age / stance)
       Striking (sig strikes, accuracy, head/body/leg)
       Grappling (TD%, sub attempts, control time)
       Top model features (what drove the prediction)
       Recent fights for each fighter
```

For **past predictions**, an additional `ActualResultCard` appears at the top:
- 🟢 Green border + "✓ Correct" — model called it right
- 🔴 Red border + "✗ Incorrect" — model was wrong
- 🟡 Amber border + "~ Upset" — model was wrong AND the underdog won

### Events (`/events`)
Historical events list. Added Completed / Upcoming toggle (reuses `/upcoming` data) and event name search filter.

### Past Prediction Event (`/past-predictions/events/:event_id`)
Shows all model predictions for a completed event. Each row: ✓/✗/~ indicator, Fighter A vs Fighter B, predicted winner + confidence + method, actual winner + method. Rows are clickable links to the fight matchup page.

### Fighter Lookup (`/fighters` + `/fighters/:id`)
List page with search. Detail page shows tale of the tape, career record (W-L-D), fight history table.

### About (`/about`)
Personal intro page. Casual tone. Explains the project, the data source, the model approach, the tech stack.

## How a User Click Triggers the Full Stack

```
┌─────────────────────────────────────────────────────────────────────┐
│  USER ACTION                                                        │
│  e.g. opens /upcoming, clicks "UFC 315" accordion row               │
└──────────────────────────┬──────────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────────┐
│  REACT (Vercel CDN)                                                 │
│                                                                     │
│  UpcomingPage.tsx                                                   │
│  └── upcomingService                                                │
│      └── apiClient.get('/upcoming/events/{id}')   ← Axios           │
│          └── VITE_API_BASE_URL + '/upcoming/events/{id}'            │
└──────────────────────────┬──────────────────────────────────────────┘
                           │  HTTP GET (JSON)
                           ▼
┌─────────────────────────────────────────────────────────────────────┐
│  FASTAPI (Google Cloud Run — Docker container)                      │
│                                                                     │
│  api/v1/endpoints/upcoming.py                                       │
│  └── get_upcoming_event_detail(event_id, db)                        │
│      ├── SQLAlchemy Session                                         │
│      └── text("SELECT ... FROM upcoming_fights                      │
│               JOIN upcoming_predictions ...")                       │
└──────────────────────────┬──────────────────────────────────────────┘
                           │  SQL query (psycopg2 / SQLAlchemy)
                           ▼
┌─────────────────────────────────────────────────────────────────────┐
│  POSTGRESQL (Supabase)                                              │
│                                                                     │
│  upcoming_fights  (scraped from UFCStats /upcoming)                 │
│  upcoming_predictions  (pre-computed by compute_predictions.py)     │
│  fighter_details, fighter_tott  (historical fighter data)           │
└──────────────────────────┬──────────────────────────────────────────┘
                           │  Result rows
                           ▼
                  FastAPI serialises to JSON
                  (Pydantic v2 response model)
                           │
                           ▼
                  Axios parses response
                  React re-renders accordion
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────────┐
│  USER SEES                                                          │
│  Fight card expanded — fighter names, win% bar, method chip         │
│  Clicks a fight row → navigates to /upcoming/fights/:id             │
└─────────────────────────────────────────────────────────────────────┘
```

## How a Prediction is Generated (Weekly Automation)

```
Friday 12:00 UTC
       │
       ▼
GitHub Actions: upcoming-predictions.yml
       │
       ├── run_upcoming.py
       │   ├── upcoming_scraper.py
       │   │   ├── GET ufcstats.com/statistics/events/upcoming
       │   │   ├── Parse event rows → upcoming_events (upsert)
       │   │   ├── Parse fight rows → upcoming_fights (upsert)
       │   │   └── Match fighter URLs → fighter_details.id (FK)
       │   │
       │   └── compute_predictions.py
       │       ├── For each upcoming_fight with both fighter FKs:
       │       │   ├── features/extractors.py → build_feature_vector()
       │       │   │   ├── Query fight_stats for rolling 5-fight window
       │       │   │   ├── Query fighter_tott for physical attributes
       │       │   │   └── Compute 30 differential features
       │       │   ├── Load trained RandomForest model (models/*.pkl)
       │       │   ├── model.predict_proba(features) → win_prob_a, win_prob_b
       │       │   ├── method_model.predict_proba() → ko_tko, sub, dec
       │       │   └── UPSERT into upcoming_predictions (prediction_source='backfill')
       │       └── Fights with NULL fighter FK → skipped (debuting fighters)
       │
Sunday 18:00 UTC
       │
       ▼
GitHub Actions: weekly-ufc-scraper.yml
       │  live_scraper.py scrapes all 6 tables for completed events
       │  on_success triggers →
       ▼
GitHub Actions: post-scrape-clean.yml
       │  ETL phases 1–4 (FK resolution, type parsing, derived columns)
       │  archive_completed_predictions.py  ← NEW (Task 26)
       │    └── Finds upcoming_fights where event_date < today and archived=FALSE
       │        Copies prediction snapshot to past_predictions
       │        with prediction_source='pre_fight_archive' (frozen pre-fight features)
       │        Marks upcoming_fights.archived=TRUE (soft delete)
       │  on_success triggers →
       ▼
GitHub Actions: feature-engineering.yml
       │  Rebuilds training_data.parquet from updated fight_stats
       │  on_success triggers →
       ▼
GitHub Actions: retrain.yml
       │  Retrains RandomForest + method model on new parquet
       │  Commits model .joblib files + metrics.json back to repo
       │  on_success triggers →
       ▼
GitHub Actions: deploy-backend.yml
       └── Builds new Docker image with updated model artifacts
           Pushes to Artifact Registry
           Deploys to Cloud Run (zero-downtime rolling update)
```

**Why the archive step runs before ETL completes**: predictions must be frozen using the *same* feature values the model saw before the fight. If we archived after ETL, the ETL would have already updated fighter rolling stats with post-fight data, creating look-ahead bias in the scorecard.

## GitHub Actions — All 7 Workflows

Every automated process in the project runs through GitHub Actions. Here's the full picture:

---

### 1. `daily-keepalive.yml` — Every day at **03:00 UTC**
**Purpose**: Prevent Supabase free tier from auto-pausing. Supabase pauses projects after 7 days of inactivity; this pings the DB daily to keep it alive.
**What it does**: Runs `scraper/keepalive_ping.py` — a single `SELECT 1` against the database.
**Trigger**: Cron `0 3 * * *` (3 AM UTC daily, ~11 PM ET).

---

### 2. `weekly-ufc-scraper.yml` — Every **Sunday at 18:00 UTC**
**Purpose**: Pick up new UFC events that completed over the weekend.
**What it does**: Runs `live_scraper.py` — scrapes UFCStats for any new events/fights/results/stats not already in the DB.
**On success**: Automatically triggers `post-scrape-clean.yml`.
**Trigger**: Cron `0 18 * * 0` (6 PM UTC Sunday, ~2 PM ET).

---

### 3. `post-scrape-clean.yml` — Triggered by `weekly-ufc-scraper.yml` success
**Purpose**: Clean raw scraped data and archive pre-fight predictions before features change.
**What it does** (in order):
1. Runs ETL phases 1–4 (`post_scrape_clean.py`) — FK resolution, type parsing, derived columns
2. Runs `archive_completed_predictions.py` — freezes any upcoming fight predictions that are now past events into `past_predictions` with `prediction_source='pre_fight_archive'`
**On success**: Automatically triggers `feature-engineering.yml`.
**Trigger**: `workflow_run` on `Weekly UFC Scraper` → `completed` (only runs if scraper succeeded).

---

### 4. `feature-engineering.yml` — Triggered by `post-scrape-clean.yml` success
**Purpose**: Rebuild the ML training matrix from the updated database.
**What it does**: Runs `features/run_build.py` — queries all fight stats, computes 30+ differential features, writes `training_data.parquet`. Commits updated `selected_features.json` back to the repo.
**On success**: Automatically triggers `retrain.yml`.
**Trigger**: `workflow_run` on `Post-Scrape ETL Cleanup` → `completed`.

---

### 5. `retrain.yml` — Triggered by `feature-engineering.yml` success
**Purpose**: Retrain the ML models on fresh data.
**What it does**: Runs `ml/run_train.py` — trains `RandomForestClassifier` (win/loss) and method classifier on the new parquet. Commits updated `.joblib` model files, `metrics.json`, `feature_importance.json`, and SHAP summary back to the repo.
**On success**: The commit to `main` triggers `deploy-backend.yml` (via the `paths: backend/**` filter).
**Trigger**: `workflow_run` on `Feature Engineering Pipeline` → `completed`.

---

### 6. `upcoming-predictions.yml` — Every **Friday at 12:00 UTC**
**Purpose**: Compute fresh predictions for the upcoming weekend's fight card.
**What it does**: Runs `scraper/run_upcoming.py` — scrapes UFCStats `/upcoming` page, upserts `upcoming_events` and `upcoming_fights`, matches fighter URLs to `fighter_details.id`, then runs `compute_predictions.py` to compute and store predictions in `upcoming_predictions`.
**Trigger**: Cron `0 12 * * 5` (12 PM UTC Friday, ~8 AM ET). Runs before weekend events.

---

### 7. `deploy-backend.yml` — On push to `main` (backend files only)
**Purpose**: Auto-deploy backend changes to Google Cloud Run.
**What it does**: Authenticates to GCP using the `GCP_SA_KEY` secret, builds the Docker image, pushes to Artifact Registry (`us-central1-docker.pkg.dev/kabes-maybes/kabes-maybes/api`), deploys to Cloud Run with a zero-downtime rolling update.
**Path filter**: Only runs when `backend/**`, `Dockerfile`, `.dockerignore`, or the workflow file itself changes — pure frontend pushes skip it entirely.
**Trigger**: `push` to `main`.

---

### Full weekly timeline

```
Friday  12:00 UTC  →  upcoming-predictions.yml  (fresh fight card predictions)
Sunday  18:00 UTC  →  weekly-ufc-scraper.yml     (scrape completed events)
                   →  post-scrape-clean.yml       (ETL + archive predictions)
                   →  feature-engineering.yml     (rebuild training matrix)
                   →  retrain.yml                 (retrain models, commit artifacts)
                   →  deploy-backend.yml          (deploy new model to Cloud Run)

Daily   03:00 UTC  →  daily-keepalive.yml         (keep Supabase alive)
```

The entire Sunday chain from scrape → retrain → deploy typically takes **15–25 minutes** end to end.

## Database Schema (Final State)

```sql
-- Core historical tables
event_details (756 rows)
    id VARCHAR(6) PK
    "EVENT", "URL", date_proper, "LOCATION"

fighter_details (4,449 rows)
    id VARCHAR(6) PK
    "FIRST", "LAST", "NICKNAME", "URL"

fight_details (8,482 rows)
    id VARCHAR(6) PK
    event_id → event_details
    fighter_a_id, fighter_b_id → fighter_details
    "BOUT", "URL"

fight_results (8,482 rows — one per fight)
    id VARCHAR(6) PK
    fight_id → fight_details, event_id → event_details
    fighter_id → fighter_details  (WINNER)
    opponent_id → fighter_details (LOSER)
    "METHOD", "ROUND", "TIME", "WEIGHTCLASS"
    fight_time_seconds, total_fight_time_seconds  (parsed)
    weight_class, is_title_fight, is_interim_title (derived)

fighter_tott (4,435 rows)  -- Tale of the Tape
    id VARCHAR(6) PK
    fighter_id → fighter_details
    height_inches, weight_lbs, reach_inches, dob_date (parsed)
    "STANCE"

fight_stats (39,912 rows — one per fighter per round)
    id VARCHAR(6) PK
    fight_id → fight_details, fighter_id → fighter_details
    sig_str_landed, sig_str_attempted, ctrl_seconds, kd_int (parsed)
    "ROUND", "KD", "TD", "SUB.ATT", "REV."

-- Upcoming / prediction tables
upcoming_events    (event_name, date_proper, location, ufcstats_url)
upcoming_fights    (event_id, fighter_a/b_id, weight_class, is_title_fight)
upcoming_predictions (fight_id UNIQUE, win_prob_a/b, method probs, features_json JSONB)

-- Past predictions (model scorecard)
past_predictions   (fight_id VARCHAR(8), event_id VARCHAR(6), all prediction + actual columns)
```

> **Note on ID sizes**: Core table IDs are `VARCHAR(6)` (UFCStats alphanumeric IDs). Fight IDs in `fight_details` and `fight_stats` are 8-char hex (e.g. `3738135e`). This was discovered when backfilling `past_predictions` caused `StringDataRightTruncation` errors — the table was rebuilt with `VARCHAR(8)` for fight FK columns.

## Prediction Archive & Data Leakage (Task 26)

### The Problem

While reviewing the model scorecard, a data leakage issue was identified. Holloway vs Oliveira had shown a **60.1% pre-fight prediction** for Holloway on the upcoming fights page — but after the fight completed and the model retrained on the new data, the scorecard was showing **50.8%** for the same fight. The scorecard number was wrong because:

1. The fight completes → Sunday automation runs → post-fight stats added to `fight_stats`
2. `feature_engineering.py` rebuilds `training_data.parquet` with the new stats (including this fight)
3. `retrain.yml` retrains the model on this updated data
4. `compute_past_predictions.py` re-runs `build_feature_vector()` using the **now-updated** rolling stats
5. The new model, trained partially on this fight's outcome, re-evaluates the same fight — and produces a different (often less extreme) probability

This is **look-ahead bias**: the scorecard was showing what the model *would have predicted* if it had trained on post-fight data, not what it *actually predicted* before the fight.

### The Fix

**`backend/db/migrations/002_prediction_archive.sql`** adds:
- `prediction_source TEXT CHECK(... IN ('pre_fight_archive', 'backfill'))` to `past_predictions`
- `pre_fight_predicted_at TIMESTAMPTZ` — when the original prediction was made
- `archived BOOLEAN DEFAULT FALSE` to `upcoming_fights` — soft-delete flag
- Unique constraint on `(fight_id, prediction_source)` so both source types can coexist

**`backend/scraper/archive_completed_predictions.py`** (runs in `post-scrape-clean.yml` before ETL):
- Queries `upcoming_fights` where `event_date < today AND archived = FALSE`
- Joins with `upcoming_predictions` for the stored prediction values
- Inserts a frozen snapshot into `past_predictions` with `prediction_source = 'pre_fight_archive'`
- These probabilities are the exact values that were shown to users before the fight — untouched by any subsequent retrain
- Sets `upcoming_fights.archived = TRUE`

**`backend/api/v1/endpoints/past_predictions.py`** — all queries now use a `DISTINCT ON (fight_id)` CTE that prefers `pre_fight_archive` over `backfill`. The scorecard only ever shows the pre-fight number when one is available.

**Frontend**: fight detail pages show a provenance badge — green "pre-fight prediction" (frozen) or amber "legacy backfill" (retroactive, pre-Task 26 fights).

### Backfill vs Archive

| | `backfill` | `pre_fight_archive` |
|---|---|---|
| Source | `compute_past_predictions.py` (retroactive) | `archive_completed_predictions.py` (real-time) |
| Features used | Current rolling stats (may include post-fight data) | Stats as-of the Friday before the fight |
| Model version | Latest retrained model | Model that was live when the fight card was posted |
| Displayed on scorecard | Only if no archive exists | Always preferred |
| Fights covered | All historical fights | Fights from Task 26 deployment onward |

## Key Engineering Decisions & Lessons

| Decision | Why |
|----------|-----|
| Raw SQL via `text()` instead of ORM models | Schema is append-only and legacy-shaped; ORMs would add complexity without benefit |
| VARCHAR IDs (not integers) | UFCStats hex IDs are natural PKs; converting to int would break scraper FK resolution |
| `fight_id = VARCHAR(8)` in past_predictions | Fight IDs are 8-char hex (e.g. `3738135e`); original table used VARCHAR(6) which caused truncation |
| Pydantic v2 for all API responses | Automatic validation, free OpenAPI docs, type-safe frontend |
| Axios interceptor error normalisation | FastAPI 422 errors return `detail` as array; guard `typeof rawDetail === 'string'` prevents `[object Object]` |
| `useRef` for debounce timers | `setTimeout` in a closure captures stale state; ref persists across renders |
| `div+onClick` instead of nested `<Link>` | `<a>` inside `<a>` is invalid HTML; browsers break the outer link unpredictably |
| `Promise.resolve(null)` in `useApi` | Skip API call when a tab is inactive without breaking the hook's dependency array |
| Google Cloud Run over Render | Render free tier cold start ~30s; Cloud Run cold start ~1–2s; both ~free at portfolio traffic |
| `min-instances=0` on Cloud Run | Scales to zero between requests (free); acceptable for a portfolio project |
| `prediction_source` column + archive script | Pre-fight predictions frozen before post-fight stats update features; prevents model scorecard look-ahead bias |
| `DISTINCT ON (fight_id)` preferring `pre_fight_archive` | Each fight has at most one backfill row and one archive row; dedup on read is simpler than enforcing it on write |
| FastAPI route order matters | `/fights` must be registered before `/fights/{fight_id}` — FastAPI matches in registration order |
| `allow_origin_regex` in CORSMiddleware | Vercel generates dynamic preview URLs per branch; regex `kabes-maybes(-git-.*-kibbbles-projects)?\.vercel\.app` covers all of them without hardcoding each |

## What's Next

1. **Fight Outcome Predictor (Task 8)**: Interactive sliders — adjust fighter attributes and see win probability update in real time. The `/predictions/fight-outcome` endpoint already exists; this is purely a frontend feature.

2. **Style Evolution Timeline (Task 9)**: Finish rate trends by year and weight class. Are KOs getting rarer? Is wrestling dominant now? The `/analytics/style-evolution` endpoint is live.

3. **Fighter Endurance Dashboard (Task 10)**: Round-by-round performance profiles. Which fighters fade in the championship rounds? The `/analytics/fighter-endurance/{id}` endpoint is live.

4. **Custom domain**: `kabesmaybes.com` or similar — point DNS at Vercel, done.